<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/Time_Interval_Hypothesis_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DOING EXPERIMENT NO.2

HYPOTHESIS _ 1 (NOT MUCH SUPPORTED)  TIME INTERVAL HYPOTHESIS

In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 1
# ALIGN VALIDATION METADATA + INSPECT TIME_INTERVAL
# ============================================================

import pandas as pd
import numpy as np
import h5py

BASE_RESULTS = "/content/drive/MyDrive/ECG_Project/results"

VAL_SPLIT = f"{BASE_RESULTS}/dataset_split/validation_split.csv"
VAL_H5    = f"{BASE_RESULTS}/preprocessing/processed/validation.h5"

val_meta = pd.read_csv(VAL_SPLIT)

# --------------------------------------
# Normalize record IDs
# --------------------------------------

val_meta["record_id_norm"] = (
    val_meta["ecg_row_record"]
    .astype(str)
    .str.replace(".dat", "", regex=False)
    .str.strip()
)

with h5py.File(VAL_H5, "r") as f:
    h5_ids = [
        x.decode() if isinstance(x, bytes) else str(x)
        for x in f["record_ids"][:]
    ]

h5_ids = [x.strip() for x in h5_ids]

# --------------------------------------
# Align metadata EXACTLY to H5 order
# --------------------------------------

val_meta_aligned = (
    val_meta
    .set_index("record_id_norm")
    .reindex(h5_ids)
)

assert val_meta_aligned["OMI"].isna().sum() == 0
assert val_meta_aligned["Time_Interval"].isna().sum() == 0

print("Validation ECGs:", len(val_meta_aligned))

print("\nTime interval statistics (minutes):")
print(
    val_meta_aligned["Time_Interval"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
)

print("\nMinimum:", val_meta_aligned["Time_Interval"].min())
print("Maximum:", val_meta_aligned["Time_Interval"].max())

print("\nNegative intervals:",
      (val_meta_aligned["Time_Interval"] < 0).sum())

print("\nOMI cases:",
      int(val_meta_aligned["OMI"].sum()))

print("non-OMI cases:",
      int((val_meta_aligned["OMI"] == 0).sum()))

Validation ECGs: 2685

Time interval statistics (minutes):
count    2685.000000
mean     2540.657728
std      2374.560324
min         5.000000
10%       191.400000
25%       453.000000
50%      1659.000000
75%      4045.000000
90%      5988.000000
95%      7431.600000
max      9989.000000
Name: Time_Interval, dtype: float64

Minimum: 5
Maximum: 9989

Negative intervals: 0

OMI cases: 166
non-OMI cases: 2519


In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 2
# CREATE TIME BINS
# ============================================================

time_bins = [
    0,
    6 * 60,        # 6 hours
    12 * 60,       # 12 hours
    24 * 60,       # 24 hours
    72 * 60,       # 72 hours
    7 * 24 * 60 + 1
]

time_labels = [
    "<6 h",
    "6–12 h",
    "12–24 h",
    "24–72 h",
    "72 h–7 d"
]

val_meta_aligned["time_group"] = pd.cut(
    val_meta_aligned["Time_Interval"],
    bins=time_bins,
    labels=time_labels,
    right=False
)

summary = (
    val_meta_aligned
    .groupby("time_group", observed=False)["OMI"]
    .agg(
        Total="count",
        OMI="sum"
    )
)

summary["non_OMI"] = (
    summary["Total"] - summary["OMI"]
)

summary["OMI_prevalence"] = (
    summary["OMI"] / summary["Total"]
)

print(summary)

            Total  OMI  non_OMI  OMI_prevalence
time_group                                     
<6 h          550   73      477        0.132727
6–12 h        248   20      228        0.080645
12–24 h       365   18      347        0.049315
24–72 h       950   36      914        0.037895
72 h–7 d      572   19      553        0.033217


In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 3
# LOAD BEST LAST2 MODEL
# ============================================================

BEST_MODEL_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "OMI_model/ecgfounder_omi_last2.pth"
)

# Build OMI architecture directly
time_model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[
        64, 160, 160,
        400, 400,
        1024, 1024
    ],
    m_blocks_list=[
        2, 2, 2,
        3, 3,
        4, 4
    ],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=1,
    use_bn=False,
    use_do=False,
    return_features=False,
    verbose=False
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

time_model.load_state_dict(
    checkpoint["model_state_dict"]
)

time_model = time_model.to(DEVICE)

time_model.eval()

print("Loaded checkpoint:")
print("Epoch:", checkpoint["epoch"])
print("Saved AUPRC:", checkpoint["val_auprc"])
print("Saved AUROC:", checkpoint["val_auroc"])

Loaded checkpoint:
Epoch: 6
Saved AUPRC: 0.4000248638802949
Saved AUROC: 0.8788891174065057


In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 4
# GET VALIDATION PREDICTIONS
# ============================================================

result = evaluate_model(
    model=time_model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    threshold=0.5
)

probs = result["probabilities"]
labels = result["labels"]

print("Predictions:", len(probs))
print("Metadata:", len(val_meta_aligned))

assert len(probs) == len(val_meta_aligned)

# Extra safety check
metadata_labels = (
    val_meta_aligned["OMI"]
    .astype(int)
    .to_numpy()
)

assert np.array_equal(
    labels,
    metadata_labels
), "Prediction/metadata order mismatch!"

print("✓ Prediction order verified.")

Predictions: 2685
Metadata: 2685
✓ Prediction order verified.


In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 5
# METRICS BY ECG → ANGIOGRAPHY INTERVAL
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    precision_score
)

analysis_df = pd.DataFrame({
    "OMI": labels,
    "probability": probs,
    "Time_Interval":
        val_meta_aligned["Time_Interval"].to_numpy(),
    "time_group":
        val_meta_aligned["time_group"].to_numpy()
})


def calculate_group_metrics(group, threshold=0.5):

    y_true = group["OMI"].to_numpy().astype(int)
    y_prob = group["probability"].to_numpy()

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    n = len(y_true)
    positives = y_true.sum()
    negatives = n - positives

    # Need both classes for AUROC
    if positives > 0 and negatives > 0:
        auroc = roc_auc_score(
            y_true,
            y_prob
        )

        auprc = average_precision_score(
            y_true,
            y_prob
        )

    else:
        auroc = np.nan
        auprc = np.nan

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "N": n,
        "OMI": positives,
        "non_OMI": negatives,
        "Prevalence": positives / n,
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1": f1,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn
    }


rows = []

for group_name in time_labels:

    group = analysis_df[
        analysis_df["time_group"] == group_name
    ]

    if len(group) == 0:
        continue

    metrics = calculate_group_metrics(group)

    metrics["Time_Group"] = group_name

    rows.append(metrics)


time_results = pd.DataFrame(rows)

time_results = time_results[
    [
        "Time_Group",
        "N",
        "OMI",
        "non_OMI",
        "Prevalence",
        "AUROC",
        "AUPRC",
        "F1",
        "Precision",
        "Sensitivity",
        "Specificity",
        "TP",
        "FN",
        "FP",
        "TN"
    ]
]

print(time_results.to_string(index=False))

Time_Group   N  OMI  non_OMI  Prevalence    AUROC    AUPRC       F1  Precision  Sensitivity  Specificity  TP  FN  FP  TN
      <6 h 550   73      477    0.132727 0.852302 0.533512 0.476190   0.364964     0.684932     0.817610  50  23  87 390
    6–12 h 248   20      228    0.080645 0.826754 0.402412 0.303797   0.203390     0.600000     0.793860  12   8  47 181
   12–24 h 365   18      347    0.049315 0.866314 0.401871 0.363636   0.250000     0.666667     0.896254  12   6  36 311
   24–72 h 950   36      914    0.037895 0.917062 0.325892 0.372414   0.247706     0.750000     0.910284  27   9  82 832
  72 h–7 d 572   19      553    0.033217 0.864281 0.202687 0.219780   0.138889     0.526316     0.887884  10   9  62 491


In [ ]:
# ============================================================
# TIME EXPERIMENT — CELL 6
# CUMULATIVE TIME WINDOWS
# ============================================================

windows = {
    "≤6 h": 6 * 60,
    "≤12 h": 12 * 60,
    "≤24 h": 24 * 60,
    "≤72 h": 72 * 60,
    "≤7 d": 7 * 24 * 60
}

rows = []

for name, max_minutes in windows.items():

    group = analysis_df[
        analysis_df["Time_Interval"] <= max_minutes
    ]

    metrics = calculate_group_metrics(group)

    metrics["Window"] = name

    rows.append(metrics)


cumulative_results = pd.DataFrame(rows)

cumulative_results = cumulative_results[
    [
        "Window",
        "N",
        "OMI",
        "non_OMI",
        "Prevalence",
        "AUROC",
        "AUPRC",
        "F1",
        "Precision",
        "Sensitivity",
        "Specificity"
    ]
]

print(cumulative_results.to_string(index=False))

Window    N  OMI  non_OMI  Prevalence    AUROC    AUPRC       F1  Precision  Sensitivity  Specificity
  ≤6 h  551   73      478    0.132486 0.852553 0.533498 0.476190   0.364964     0.684932     0.817992
 ≤12 h  798   93      705    0.116541 0.846671 0.496908 0.429066   0.316327     0.666667     0.809929
 ≤24 h 1163  111     1052    0.095443 0.857269 0.477979 0.416901   0.303279     0.666667     0.838403
 ≤72 h 2114  147     1967    0.069536 0.880383 0.435785 0.403194   0.285311     0.687075     0.871378
  ≤7 d 2685  166     2519    0.061825 0.878889 0.400025 0.375635   0.261176     0.668675     0.875347
